# Validation

How do we know the estimator is correct and not 'a big pile of code that
returns nonsense'? Two independent checks.

1. **Trivial limit:** with thresholds pushed beyond the data range,
   nothing is censored, so the censored/truncated MLE *must* reduce to
   ordinary least squares. We verify this against `statsmodels.OLS`.
2. **Reference packages:** the test suite also compares against R's
   `AER::tobit` and `truncreg` (run when R is available); see
   `tests/test_r_reference.py`.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from censtrunc import CensoredRegression, TruncatedRegression

rng = np.random.default_rng(0)
n = 1000
X = rng.normal(size=(n, 3))
y = 2.0 + 1.5*X[:,0] - 0.7*X[:,1] + 0.3*X[:,2] + rng.normal(scale=1.3, size=n)

## No censoring $\Rightarrow$ OLS

We set `left=-1e6`, `right=1e6` so that no observation is censored. The censored log-likelihood then reduces to the Gaussian log-likelihood, whose maximiser is exactly the OLS coefficient vector.

In [2]:
ols = sm.OLS(y, sm.add_constant(X)).fit()
cens = CensoredRegression(left=-1e6, right=1e6).fit(X, y)
trunc = TruncatedRegression(left=-1e6, right=1e6).fit(X, y)

pd.DataFrame({
    'OLS':            np.asarray(ols.params),
    'Censored MLE':   cens.coef_,
    'Truncated MLE':  trunc.coef_,
}, index=['const', 'x1', 'x2', 'x3']).round(6)

,OLS,Censored MLE,Truncated MLE
const,2.046512,2.046482,2.046512
x1,1.434191,1.434221,1.434191
x2,-0.626598,-0.626603,-0.626598
x3,0.345980,0.345982,0.345980


The columns agree to several decimals. We can quantify the maximum discrepancy and confirm the scale parameter and log-likelihood match too (using the MLE scale $\hat\sigma = \sqrt{\mathrm{SSR}/n}$).

In [3]:
ols_beta = np.asarray(ols.params)
print(f'max |beta_censored - beta_OLS|  = {np.max(np.abs(cens.coef_ - ols_beta)):.2e}')
print(f'max |beta_truncated - beta_OLS| = {np.max(np.abs(trunc.coef_ - ols_beta)):.2e}')
print(f'|sigma_censored - sqrt(SSR/n)|  = {abs(cens.sigma_ - np.sqrt(ols.ssr/n)):.2e}')
print(f'|loglik_censored - loglik_OLS|  = {abs(cens.llf_ - ols.llf):.2e}')

max |beta_censored - beta_OLS|  = 3.00e-05
max |beta_truncated - beta_OLS| = 4.44e-16
|sigma_censored - sqrt(SSR/n)|  = 7.55e-07
|loglik_censored - loglik_OLS|  = 5.90e-07


All discrepancies are at the level of the optimiser tolerance — the estimator behaves exactly as theory requires in the no-censoring limit, which is strong evidence the likelihood and its optimisation are implemented correctly.

## With censoring: agreement with R, disagreement with OLS

The decisive test is a genuinely censored dataset. We compare four numbers per coefficient:

- **true** — the data-generating value;
- **censtrunc** — this package's MLE;
- **R survreg** — R's Gaussian `survreg`, the engine behind `AER::tobit`, computed by an entirely independent codebase;
- **OLS** — naive least squares, which is biased under censoring.

Two correct maximum-likelihood implementations must converge to the *same unique* optimum, so `censtrunc` and R should agree to optimiser tolerance — while both should differ from the biased OLS and sit close to the truth.

In [4]:
import shutil, subprocess, json, tempfile, os
from pathlib import Path

rng2 = np.random.default_rng(20260527)
nc = 4000
Xc = rng2.normal(size=(nc, 2))
y_star_c = 1.0 + 0.7*Xc[:, 0] - 0.4*Xc[:, 1] + rng2.normal(size=nc)
Lc, Rc = 0.0, 2.5
yc = np.clip(y_star_c, Lc, Rc)

cens = CensoredRegression(left=Lc, right=Rc).fit(Xc, yc)
ols_c = np.asarray(sm.OLS(yc, sm.add_constant(Xc)).fit().params)

# Run R's survreg via the bundled reference script, if R is available.
def _find_r_script():
    for c in [Path('tests/reference/fit_tobit.R'),
              Path('../tests/reference/fit_tobit.R')]:
        if c.exists():
            return c
    return None

r_beta = r_sigma = None
rscript, script_path = shutil.which('Rscript'), _find_r_script()
if rscript and script_path:
    fd, csvp = tempfile.mkstemp(suffix='.csv'); os.close(fd)
    np.savetxt(csvp, np.column_stack([yc, Xc]), delimiter=',', header='y,x1,x2', comments='')
    try:
        out = subprocess.run([rscript, str(script_path), csvp, str(Lc), str(Rc)],
                             capture_output=True, text=True, timeout=120, check=True)
        rt = json.loads(out.stdout)['tobit']
        r_beta = np.array([rt['coef']['(Intercept)'], rt['coef']['x1'], rt['coef']['x2']])
        r_sigma = float(rt['scale'])
    except Exception:
        r_beta = None
    finally:
        os.remove(csvp)

In [5]:
if r_beta is not None:
    table = pd.DataFrame({
        'true':         [1.0, 0.7, -0.4],
        'censtrunc':    cens.coef_,
        'R survreg':    r_beta,
        'OLS (biased)': ols_c,
    }, index=['const', 'x1', 'x2'])
    display(table.round(6))
    print(f'max |censtrunc - R|  = {np.max(np.abs(cens.coef_ - r_beta)):.2e}  (independent solvers, same optimum)')
    print(f'|sigma_censtrunc - sigma_R| = {abs(cens.sigma_ - r_sigma):.2e}')
    print(f'max |censtrunc - OLS| = {np.max(np.abs(cens.coef_ - ols_c)):.3f}  (Tobit correction is real)')
else:
    print('R not available at build time.')
    print('The live comparison runs in the test suite: tests/test_r_reference.py')
    print('censtrunc estimates:', cens.coef_.round(4))
    print('OLS (biased):       ', ols_c.round(4))

,true,censtrunc,R survreg,OLS (biased)
const,1.0,1.014284,1.014284,1.090364
x1,0.7,0.701331,0.701331,0.470177
x2,-0.4,-0.410105,-0.410105,-0.277838


max |censtrunc - R|  = 3.05e-08  (independent solvers, same optimum)
|sigma_censtrunc - sigma_R| = 2.41e-08
max |censtrunc - OLS| = 0.231  (Tobit correction is real)


The `censtrunc` and R columns are identical to about six decimals, yet `max |censtrunc - R|` is a tiny *non-zero* number (~1e-8): the two independent optimisers land on the same maximum without being bit-for-bit copies. Both recover the true coefficients, while OLS is visibly biased toward zero — exactly the censoring attenuation predicted by Greene (1981).